# 02 · 데이터 전처리 (Single Source of Truth)

> **이 노트북의 역할**: 원본 CSV를 깨끗하게 다듬고 파생변수를 붙인 뒤,
> **train / val / test 세 덩어리로 '얼려서' 저장**한다.
> 이후 모든 모델 노트북(03~08)은 *전처리를 다시 하지 않고* 여기서 만든 파일만 불러 쓴다.
> 전처리를 여기 한 곳에 모으면 = 모두가 똑같은 데이터를 본다 = **공정한 비교 + 재현성**(루브릭 직결).

**파이프라인**: 로드 → 정제 → 누수점검 → 파생변수 → log1p → 분할 → 결측대치 → 표준화 → 저장 → 검증
- 누수 점검: 모델이 학습 도중 미래의 정보나 Target 힌트를 미리 알아채지 못하도록 차단하는 과정  

## 0. 환경 설정

라이브러리·경로를 잡고 난수를 고정한다.
- **pandas**: 표(엑셀 같은 2차원 데이터)를 다루는 파이썬 표준 도구. 불러오기·가공·저장 전반을 담당.
- **scikit-learn**: 대표적 머신러닝 라이브러리. 여기선 *분할(train_test_split)* 과 *표준화(StandardScaler)* 만 빌려 씀.
- **set_seed(42)**: 무작위 분할이 매번 같게 나오도록 '주사위'를 고정 → 재현성.

In [1]:
# autoreload: utils.py를 고치면 커널 재시작 없이 자동 반영
%load_ext autoreload
%autoreload 2

from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib

from utils import set_seed          # 우리가 만든 공유 모듈
set_seed(42)

# ── 경로: 이 한 곳만 고치면 다른 PC에서도 그대로 돌아감 ──
PROJECT_ROOT = Path(r"C:\Users\Administrator\Desktop\딥러닝응용\TermProject")
DATA_DIR  = PROJECT_ROOT / "dataset_1"
OUT_DIR   = PROJECT_ROOT / "processed"
OUT_DIR.mkdir(exist_ok=True)
TRAIN_CSV = DATA_DIR / "Kaggle_Training_Dataset_v2.csv"
TEST_CSV  = DATA_DIR / "Kaggle_Test_Dataset_v2.csv"

print("데이터 존재 확인 →", "train:", TRAIN_CSV.exists(), "/ test:", TEST_CSV.exists())

데이터 존재 확인 → train: True / test: True


## 1. 원본 로드

CSV를 불러온다. 이 데이터는 **맨 끝에 값이 전부 빈 꼬리 행**이 하나 붙어 있어(원본 특성) 떼어냄.

In [2]:
def load_csv(path):
    df = pd.read_csv(path, low_memory=False)
    # 마지막 행이 절반 이상 NaN이면 '빈 꼬리 행' → 제거
    if df.iloc[-1].isnull().sum() > len(df.columns) // 2:
        df = df.iloc[:-1].copy()
    return df

df_train_raw = load_csv(TRAIN_CSV)
df_test_raw  = load_csv(TEST_CSV)
print(f"train: {df_train_raw.shape}   test: {df_test_raw.shape}")
df_train_raw.head(3)

train: (1687860, 23)   test: (242075, 23)


,sku,national_inv,lead_time,in_transit_qty,forecast_3_month,forecast_6_month,forecast_9_month,sales_1_month,sales_3_month,sales_6_month,...,pieces_past_due,perf_6_month_avg,perf_12_month_avg,local_bo_qty,deck_risk,oe_constraint,ppap_risk,stop_auto_buy,rev_stop,went_on_backorder
0,1026827,0.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,-99.00,-99.00,0.0,No,No,No,Yes,No,No
1,1043384,2.0,9.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.99,0.99,0.0,No,No,No,Yes,No,No
2,1043696,2.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,-99.00,-99.00,0.0,Yes,No,No,Yes,No,No


## 2. 정제 (행 단위)

세 가지를 정리한다. 이 단계는 **한 행만 보고 처리**하므로 분할 전에 해도 안전(다른 행을 훔쳐보지 않음).
1. **Yes/No → 1/0**: 모델은 글자를 못 먹으니 숫자로.
2. **perf의 −99 → 결측(NaN)**: −99는 실제 성과치가 아니라 '측정 불가'라는 *코드값*. 숫자로 두면 모델이 "성과 −99"로 오해함.
3. **결측 플래그 생성**: '값이 없다'는 사실 자체가 신호일 수 있어, 지우기 전에 `perf_missing`·`lead_time_missing`로 기록.
4. **불필요 컬럼 제거**: `sku`(식별자), `rev_stop`·`stop_auto_buy`(EDA에서 분별력 0으로 확인).

In [3]:
BINARY_COLS = ['potential_issue','deck_risk','oe_constraint','ppap_risk',
               'stop_auto_buy','rev_stop','went_on_backorder']
DROP_COLS   = ['sku', 'rev_stop', 'stop_auto_buy']

def clean_rowwise(df):
    df = df.copy()
    # Yes/No → 1/0
    for c in BINARY_COLS:
        if c in df.columns:
            df[c] = (df[c] == 'Yes').astype(int)
    # perf의 -99(측정불가 코드) → 결측
    for c in ['perf_6_month_avg', 'perf_12_month_avg']:
        df[c] = df[c].replace(-99, np.nan)
    # '결측이라는 사실'을 플래그로 보존 (impute 전에 만들어야 함)
    df['perf_missing']      = df['perf_6_month_avg'].isnull().astype(int)
    df['lead_time_missing'] = df['lead_time'].isnull().astype(int)
    # 불필요 컬럼 제거
    df = df.drop(columns=[c for c in DROP_COLS if c in df.columns])
    return df

df_train = clean_rowwise(df_train_raw)
df_test  = clean_rowwise(df_test_raw)
print("정제 후 컬럼 수:", df_train.shape[1])

정제 후 컬럼 수: 22


## 3. 타겟 누수(leakage) 점검

**누수** = 정답을 미리 훔쳐보는 변수가 피처에 섞이는 것. 그러면 시험 성적은 좋아도 실전에선 폭망.
`local_bo_qty`(현재 백오더 수량)처럼 **타겟과 거의 같은 뜻일 수 있는** 변수를 점검한다.
**lift** = (그 조건일 때 백오더율) ÷ (전체 백오더율). 수십 배로 튀면 의심 신호.
> 여기선 *제거하지 않고* 수치만 확인해 둔다 — 보고서 "한계/논의"의 거리. (지우는 건 팀 합의로 결정)

In [4]:
base = df_train['went_on_backorder'].mean()
print(f"전체 백오더율: {base:.3%}\n")
checks = {
    'local_bo_qty > 0'  : df_train['local_bo_qty'] > 0,
    'pieces_past_due > 0': df_train['pieces_past_due'] > 0,
    'national_inv < 0'  : df_train['national_inv'] < 0,
}
for name, mask in checks.items():
    rate = df_train.loc[mask, 'went_on_backorder'].mean()
    print(f"{name:20s} 백오더율 {rate:7.2%}  →  lift {rate/base:5.1f}x")

전체 백오더율: 0.669%

local_bo_qty > 0     백오더율   5.99%  →  lift   9.0x
pieces_past_due > 0  백오더율   4.14%  →  lift   6.2x
national_inv < 0     백오더율  15.12%  →  lift  22.6x


## 4. 파생변수 (행 단위)

원본 숫자들을 조합해 **'의미 있는 신호'**로 가공한다.
예: 재고가 있어도 안전재고·미납분을 빼면 '진짜 여유'는 마이너스일 수 있음.
- 가용재고 · 실질가용재고 · 3개월 미래가용재고 · 부족률  (SCM 도메인 논리)
- 재고마이너스 · 지역백오더 · 지연부품 플래그  (EDA의 lift 분석에서 강했던 신호)

모두 **한 행 안에서** 계산되므로 분할 전 처리해도 안전.

In [5]:
def add_features(df):
    df = df.copy()
    df['available_inventory']           = df['national_inv'] - df['min_bank']
    df['real_available_inventory']      = df['national_inv'] - df['min_bank'] - df['local_bo_qty']
    df['future_available_inventory_3m'] = (df['national_inv'] + df['in_transit_qty']
                                           - df['min_bank'] - df['forecast_3_month'])
    df['shortage_ratio_3m']             = ((df['forecast_3_month'] - df['national_inv'] - df['in_transit_qty'])
                                           / (df['forecast_3_month'] + 1))
    df['no_sales_but_demand_signal']    = (((df['sales_6_month'] == 0)
                                            & ((df['forecast_3_month'] > 0) | (df['local_bo_qty'] > 0)))
                                           .astype(int))
    df['neg_inv_flag']      = (df['national_inv'] < 0).astype(int)
    df['has_local_bo']      = (df['local_bo_qty'] > 0).astype(int)
    df['has_past_due']      = (df['pieces_past_due'] > 0).astype(int)
    df['below_safety_flag'] = (df['national_inv'] < df['min_bank']).astype(int)
    df['sales_acceleration'] = np.where(df['sales_3_month'] > 0,
                                        (df['sales_1_month'] * 3 - df['sales_3_month']) / df['sales_3_month'], 0)
    df['forecast_accuracy']  = np.where(df['forecast_3_month'] > 0,
                                        df['sales_3_month'] / df['forecast_3_month'], 0)
    df['total_risk_score']   = (df['neg_inv_flag'] * 3 + df['has_local_bo'] * 2
                                + df['potential_issue'] * 2 + df['has_past_due'] + df['below_safety_flag'])
    return df

df_train = add_features(df_train)
df_test  = add_features(df_test)
print("파생변수 추가 후 컬럼 수:", df_train.shape[1])

파생변수 추가 후 컬럼 수: 34


## 5. log1p 변환 (롱테일 압축)

수량 변수들은 대부분 **0~몇 개에 몰려 있고 가끔 수천**인 극단적 쏠림(right-skew).
`log`를 씌우면 큰 값은 눌리고 작은 값들의 차이는 살아나 학습이 안정된다.
재고 파생변수는 음수가 될 수 있어 **부호 보존** `sign(x)·log1p(|x|)`를 쓴다.
(`log1p(x) = log(1+x)` — 0도 안전하게 처리)

In [6]:
LOG_COLS = ['national_inv','in_transit_qty','forecast_3_month','forecast_6_month',
            'forecast_9_month','sales_1_month','sales_3_month','sales_6_month',
            'sales_9_month','min_bank','pieces_past_due','local_bo_qty',
            'available_inventory','real_available_inventory','future_available_inventory_3m']

def apply_log1p(df):
    df = df.copy()
    for c in LOG_COLS:
        df[c] = np.sign(df[c]) * np.log1p(np.abs(df[c]))
    return df

df_train = apply_log1p(df_train)
df_test  = apply_log1p(df_test)
print("log1p 적용 완료 (", len(LOG_COLS), "개 컬럼 )")

log1p 적용 완료 ( 15 개 컬럼 )


## 6. 분할  train / val / test

- train(학습) → 다시 **train : val = 8 : 2**. val은 '모의고사'.
- test는 **캐글이 따로 준 test 파일** 그대로(진짜 시험).
- **stratify(층화)**: 백오더가 0.67%뿐이라 그냥 자르면 val에 양성이 거의 안 들어갈 수 있음.
  타겟 비율을 train·val에 **똑같이** 유지하도록 강제.

In [7]:
TARGET = 'went_on_backorder'

train_df, val_df = train_test_split(
    df_train, test_size=0.2, random_state=42, stratify=df_train[TARGET])
test_df = df_test.copy()

for name, d in [('train', train_df), ('val', val_df), ('test', test_df)]:
    print(f"{name:5s}: {len(d):>9,}행   양성 {d[TARGET].mean():.3%}")

train: 1,350,288행   양성 0.669%
val  :   337,572행   양성 0.669%
test :   242,075행   양성 1.110%


## 7. 결측치 대치 (train 중앙값으로만)

남은 결측(lead_time, perf)을 **중앙값**으로 채운다.
핵심 원칙: **중앙값은 train에서만 계산** → val/test에 적용.
val/test는 '아직 못 본 미래'라서 걔네 통계를 미리 쓰면 그게 곧 **누수**.
(평균 대신 중앙값: 극단값에 덜 휘둘림)

In [8]:
feature_cols = [c for c in train_df.columns if c != TARGET]
medians = train_df[feature_cols].median()      # ← train에서만 계산

for d in (train_df, val_df, test_df):
    d[feature_cols] = d[feature_cols].fillna(medians)

print("남은 결측치 →",
      "train:", int(train_df.isnull().sum().sum()),
      " val:", int(val_df.isnull().sum().sum()),
      " test:", int(test_df.isnull().sum().sum()))

남은 결측치 → train: 0  val: 0  test: 0


## 8. 표준화 StandardScaler (출발선 통일)

변수마다 단위가 천차만별(재고=수천, 비율=0~1)이라 신경망이 큰 숫자에 휘둘린다.
각 변수를 **평균 0 · 표준편차 1**로 맞춰 같은 출발선에 세움.
- **StandardScaler**: scikit-learn의 표준화 도구. `fit`(평균·표준편차 학습) + `transform`(적용)의 2단계.
- 여기서도 **fit은 train만** → val/test엔 transform만. (누수 방지)
- 참고: 트리 모델(LightGBM)은 스케일 영향이 없지만, 한 데이터로 통일하려고 그대로 둠(순서 보존이라 트리 결과 불변).

In [9]:
scaler = StandardScaler()
train_df[feature_cols] = scaler.fit_transform(train_df[feature_cols])  # fit + transform
val_df[feature_cols]   = scaler.transform(val_df[feature_cols])        # transform만
test_df[feature_cols]  = scaler.transform(test_df[feature_cols])       # transform만

print("표준화 완료. 예) national_inv 평균 ~0 ->",
      round(float(train_df['national_inv'].mean()), 6))

표준화 완료. 예) national_inv 평균 ~0 -> 0.0


## 9. 저장

가공 끝난 세 덩어리를 파일로 **얼려둔다**. 이게 SSOT(단일 진실 공급원).
- **pickle(.pkl)**: 파이썬 객체(여기선 DataFrame)를 자료형까지 그대로 파일로 직렬화하는 표준 포맷.
  빠르고 타입 보존이 완벽. (단점: 파이썬·라이브러리 버전에 의존 → 팀이 같은 환경을 쓰는 전제, 외부 배포엔 부적합)
  *원래 parquet을 쓰려 했으나 이 환경에 `pyarrow`가 없어 pickle로 대체. 설치하면 `to_parquet`로 바꿔도 됨.*
- **joblib**: scaler·중앙값·피처목록을 함께 저장 → 나중에 새 데이터가 들어와도 똑같이 변환 가능.

In [10]:
train_df.to_pickle(OUT_DIR / "train.pkl")
val_df.to_pickle(OUT_DIR / "val.pkl")
test_df.to_pickle(OUT_DIR / "test.pkl")

joblib.dump({'scaler': scaler, 'medians': medians, 'feature_cols': feature_cols},
            OUT_DIR / "preprocess_artifacts.joblib")

print("저장 위치:", OUT_DIR)
print("생성 파일:", [p.name for p in sorted(OUT_DIR.glob('*'))])

저장 위치: C:\Users\Administrator\Desktop\딥러닝응용\TermProject\processed
생성 파일: ['preprocess_artifacts.joblib', 'test.pkl', 'train.pkl', 'val.pkl']


## 10. 검증

저장한 걸 **다시 불러와** 컬럼·행수·양성비율이 멀쩡한지 확인한다.
여기까지 통과하면 03~08 노트북은 `from utils import load_processed` 한 줄로 바로 시작.

In [11]:
from utils import load_processed
tr, va, te = load_processed(OUT_DIR)

assert list(tr.columns) == list(va.columns) == list(te.columns), "컬럼 불일치!"
print("재로드 OK")
print("피처 수:", len([c for c in tr.columns if c != TARGET]))
print(f"train {len(tr):,} / val {len(va):,} / test {len(te):,}")
print("양성 비율 — train {:.3%} / val {:.3%} / test {:.3%}".format(
      tr[TARGET].mean(), va[TARGET].mean(), te[TARGET].mean()))

재로드 OK
피처 수: 33
train 1,350,288 / val 337,572 / test 242,075
양성 비율 — train 0.669% / val 0.669% / test 1.110%


---
### 다음 단계
`processed/` 폴더에 **`train.pkl · val.pkl · test.pkl · preprocess_artifacts.joblib`** 생성 완료.

다음 노트북:
- **`03_Baseline_Tree`** — LightGBM 벤치마크(성능 상한선)
- **`04_MLP`** — 딥러닝 베이스라인 + 불균형 기법 ablation

모든 노트북은 이 파일들을 `load_processed(OUT_DIR)`로 불러 **동일 데이터**로 학습.